## The goal is to come up with a predictive model which helps prioritize quotes with highest chance of cross selling the product

  ### Case Details: 
  
  #### Existing customers have been quoted for "Product X" and the status of each quote is marked as Sold or Lost. The relavent data is stored in three files: 
  #### 1. `employers.csv` - holds basic customer data, such as number of years as a client, number of employees and Industry type. 
  #### 2. `geography.csv` - contains general geographic location by zip code, such as longitude, latitude, population etc... 
  #### 3. `quotes.csv` - contains the quote status (lost or sold) by the customer's ID. 
  #### 4. `dictionary.csv` - contains additional information on what each data set contains  

In [160]:
!pip install scikit-learn

In [161]:
# Import Packages

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pylab as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.cluster import KMeans


from scipy import stats
from scipy.stats import norm, skew




In [162]:
# Load Data
employer_data = pd.read_csv("data/employers.csv", encoding="cp1252")
geography_data = pd.read_csv("data/geography.csv", encoding="cp1252")
quotes_data = pd.read_csv("data/quotes.csv", encoding="cp1252")


In [176]:
# Inspect employer table
employer_data.info()
employer_data.head()

# Check for duplicates
employer_data["EmployerId"].duplicated().sum()
employer_data[employer_data["EmployerId"].duplicated(keep=False)].sort_values("EmployerId")

<class 'pandas.DataFrame'>
RangeIndex: 65356 entries, 0 to 65355
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   EmployerId       65356 non-null  int64
 1   ClientTenure     65356 non-null  int64
 2   Employees        65356 non-null  int64
 3   Industry         57125 non-null  str  
 4   NetworkStrength  65356 non-null  str  
 5   ZipCode          65250 non-null  Int64
dtypes: Int64(1), int64(3), str(2)
memory usage: 3.1 MB


,EmployerId,ClientTenure,Employees,Industry,NetworkStrength,ZipCode
29354,582410,17,9,I14,0.8253,44113
29138,582410,17,9,I27,0.8253,44113
30025,622367,20,19,I02,-0.1613,71303
30241,622367,20,19,I27,-0.1613,71303
26445,624312,11,17,I25,1.2942,92227
26229,624312,11,17,I14,1.2942,92227
24010,792438,21,61,I14,0.2819,23255
24226,792438,21,61,I27,0.2819,23255
8835,20013814,7,18,I11,0.4896,84107
8619,20013814,7,18,I02,0.4896,84107


In [164]:
geography_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 43318 entries, 0 to 43317
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   zip         43318 non-null  int64  
 1   city        43318 non-null  str    
 2   latitude    43318 non-null  float64
 3   longitude   43318 non-null  float64
 4   fips        43318 non-null  int64  
 5   county      43318 non-null  str    
 6   population  43318 non-null  int64  
 7   state       43318 non-null  str    
 8   cbsa        43318 non-null  str    
 9   cbsa_name   43318 non-null  str    
dtypes: float64(2), int64(3), str(5)
memory usage: 3.3 MB


In [165]:
quotes_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7398 entries, 0 to 7397
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   EmployerId  7398 non-null   int64
 1   Offered     7398 non-null   str  
 2   Status      7398 non-null   str  
dtypes: int64(1), str(2)
memory usage: 173.5 KB


In [ ]:
#Change Zipcode columns to same name in both tables
geography_data.rename(columns={"zip": "ZipCode"}, inplace=True)

# Change merge columns to same Dtypes (numeric)
employer_data["ZipCode"] = pd.to_numeric(
    employer_data["ZipCode"],
    errors="coerce"
).astype("Int64")

<class 'pandas.DataFrame'>
RangeIndex: 7402 entries, 0 to 7401
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   EmployerId       7402 non-null   int64  
 1   Offered          7402 non-null   str    
 2   Status           7402 non-null   str    
 3   ClientTenure     7402 non-null   int64  
 4   Employees        7402 non-null   int64  
 5   Industry         6004 non-null   str    
 6   NetworkStrength  7402 non-null   str    
 7   ZipCode          7395 non-null   Int64  
 8   city             7383 non-null   str    
 9   latitude         7383 non-null   float64
 10  longitude        7383 non-null   float64
 11  fips             7383 non-null   float64
 12  county           7383 non-null   str    
 13  population       7383 non-null   float64
 14  state            7383 non-null   str    
 15  cbsa             7383 non-null   str    
 16  cbsa_name        7383 non-null   str    
dtypes: Int64(1), float64(4), 

### Data Cleaning

In [ ]:
# Merge Dataframes
data = (quotes_data.merge(employer_data, on="EmployerId", how="left").merge(geography_data, on="ZipCode", how="left"))
data.head()
data.info()

In [167]:
# Remove Null values
data_clean = data.dropna()
data_clean.info()

<class 'pandas.DataFrame'>
Index: 5990 entries, 0 to 7401
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   EmployerId       5990 non-null   int64  
 1   Offered          5990 non-null   str    
 2   Status           5990 non-null   str    
 3   ClientTenure     5990 non-null   int64  
 4   Employees        5990 non-null   int64  
 5   Industry         5990 non-null   str    
 6   NetworkStrength  5990 non-null   str    
 7   ZipCode          5990 non-null   Int64  
 8   city             5990 non-null   str    
 9   latitude         5990 non-null   float64
 10  longitude        5990 non-null   float64
 11  fips             5990 non-null   float64
 12  county           5990 non-null   str    
 13  population       5990 non-null   float64
 14  state            5990 non-null   str    
 15  cbsa             5990 non-null   str    
 16  cbsa_name        5990 non-null   str    
dtypes: Int64(1), float64(4), int64

In [168]:
# Normalize column names
data_clean.columns = data.columns.str.lower()

# Normalize rows
data_clean["county"] = data_clean["county"].str.replace(" ", "_")
data_clean.head()

,employerid,offered,status,clienttenure,employees,industry,networkstrength,zipcode,city,latitude,longitude,fips,county,population,state,cbsa,cbsa_name
0,363773443,ProductX,Lost,4,55,I26,0.1011,77075,Houston,29.620881,-95.26018,48201.0,Harris_County,4092459.0,TX,26420,Houston-The Woodlands-Sugar Land
1,186941116,ProductX,Lost,5,10,I12,-0.0333,6002,Bloomfield,41.832798,-72.72642,9003.0,Hartford_County,894014.0,CT,25540,Hartford-West Hartford-East Hartford
2,511147993,ProductX,Lost,10,48,I19,0.8138,30067,Marietta,33.933002,-84.47633,13067.0,Cobb_County,688078.0,GA,12060,Atlanta-Sandy Springs-Roswell
3,46777684,ProductX,Lost,6,26,I20,-0.1963,2886,Warwick,41.705478,-71.45119,44003.0,Kent_County,166158.0,RI,39300,Providence-Warwick
4,115068663,ProductX,Lost,6,64,I27,0.1815,60440,Bolingbrook,41.703097,-88.07462,17197.0,Will_County,677560.0,IL,16980,Chicago-Naperville-Elgin


In [169]:
# Convert networkstrength to numeric
data_clean["networkstrength"] = pd.to_numeric(data_clean["networkstrength"], errors="coerce")

In [170]:
# Encode the target variable
data_clean["status"].unique()
data_clean["status"] = data_clean["status"].str.strip().str.lower()
data_clean["status"] = data_clean["status"].map({"sold": 1, "lost": 0})
data_clean.head()

,employerid,offered,status,clienttenure,employees,industry,networkstrength,zipcode,city,latitude,longitude,fips,county,population,state,cbsa,cbsa_name
0,363773443,ProductX,0,4,55,I26,0.1011,77075,Houston,29.620881,-95.26018,48201.0,Harris_County,4092459.0,TX,26420,Houston-The Woodlands-Sugar Land
1,186941116,ProductX,0,5,10,I12,-0.0333,6002,Bloomfield,41.832798,-72.72642,9003.0,Hartford_County,894014.0,CT,25540,Hartford-West Hartford-East Hartford
2,511147993,ProductX,0,10,48,I19,0.8138,30067,Marietta,33.933002,-84.47633,13067.0,Cobb_County,688078.0,GA,12060,Atlanta-Sandy Springs-Roswell
3,46777684,ProductX,0,6,26,I20,-0.1963,2886,Warwick,41.705478,-71.45119,44003.0,Kent_County,166158.0,RI,39300,Providence-Warwick
4,115068663,ProductX,0,6,64,I27,0.1815,60440,Bolingbrook,41.703097,-88.07462,17197.0,Will_County,677560.0,IL,16980,Chicago-Naperville-Elgin


In [171]:
# Numeric feature correlation to target variable
numeric_features = data_clean.select_dtypes(include=[np.number])
correlation_matrix = numeric_features.corr()
correlation_matrix["status"].sort_values(ascending=False)

status             1.000000
employerid         0.217305
longitude          0.063285
networkstrength    0.043771
employees         -0.023568
latitude          -0.024690
population        -0.025770
fips              -0.035232
zipcode           -0.052863
clienttenure      -0.094438
Name: status, dtype: float64